# 🚀 HUẤN LUYỆN MODEL PHÂN TÍCH HÀNH VI BẢO VỆ TRÊN KAGGLE
**Bài toán**: Nhận diện hành vi: Hút thuốc, Dùng điện thoại, Ngủ gật, Bạo lực (và hỗ trợ Bỏ vị trí trực)
**Thời hạn hoàn thành**: 30/10
**GPU khuyến nghị trên Kaggle**: Chọn Accelerator là **GPU T4 x2** hoặc **GPU P100**.

In [ ]:
# 1. KIỂM TRA GPU VÀ CÀI ĐẶT MÔI TRƯỜNG
!nvidia-smi
!pip install -q ultralytics roboflow opencv-python matplotlib

In [ ]:
# 2. IMPORT THƯ VIỆN & KIỂM TRA PYTORCH CUDA
import torch
import ultralytics
from ultralytics import YOLO
import os
import shutil

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
ultralytics.checks()

### 3. TẢI HOẶC CẤU HÌNH DATASET (ROBOFLOW / KAGGLE)
Bạn có thể dùng Roboflow API hoặc tải trực tiếp các bộ dataset đã gán nhãn sẵn về hành vi an ninh.

In [ ]:
# TẠO THƯ MỤC LƯU DATASET
dataset_dir = "/kaggle/working/dataset"
os.makedirs(dataset_dir, exist_ok=True)

# Ví dụ tải dataset từ Roboflow (Thay bằng API Key và project của bạn nếu có)
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("security-behavior").project("guard-actions")
# version = project.version(1)
# dataset = version.download("yolov8", location=dataset_dir)

# HOẶC CẤU HÌNH data.yaml CHO TẬP DỮ LIỆU ĐÃ CÓ TRÊN KAGGLE
yaml_content = """
path: /kaggle/working/dataset
train: images/train
val: images/val

names:
  0: person
  1: phone
  2: smoking
  3: sleeping
  4: violence
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(yaml_content.strip())
print("Đã khởi tạo data.yaml thành công!")

### 4. KHỞI TẠO MÔ HÌNH VÀ HUẤN LUYỆN (FINE-TUNE)
- Khuyến nghị sử dụng **YOLOv8m** hoặc **YOLO11m** (Cân bằng tốt nhất giữa độ chính xác và tốc độ realtime $\ge 25$ FPS trên camera).

In [ ]:
# Khởi tạo mô hình pre-trained COCO để transfer learning
model = YOLO('yolov8m.pt')  # Hoặc 'yolo11m.pt'

# Bắt đầu huấn luyện
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=60,             # Số epoch khuyến nghị: 50 - 100
    imgsz=640,             # Kích thước ảnh chuẩn camera giám sát
    batch=16,              # Điều chỉnh theo dung lượng VRAM của GPU Kaggle
    device=0,              # GPU 0
    workers=4,
    patience=15,           # Early stopping nếu 15 epochs không cải thiện
    save=True,
    save_period=10,
    project='/kaggle/working/runs',
    name='guard_behavior_model',
    exist_ok=True
)

### 5. ĐÁNH GIÁ KẾT QUẢ HUẤN LUYỆN
Xem độ chính xác (mAP50, mAP50-95, Precision, Recall) và ma trận nhầm lẫn (Confusion Matrix).

In [ ]:
# Đánh giá trên tập Validation
metrics = model.val()
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

# Hiển thị biểu đồ kết quả
from IPython.display import Image, display
results_png = '/kaggle/working/runs/guard_behavior_model/results.png'
confusion_matrix_png = '/kaggle/working/runs/guard_behavior_model/confusion_matrix.png'

if os.path.exists(results_png):
    display(Image(filename=results_png))
if os.path.exists(confusion_matrix_png):
    display(Image(filename=confusion_matrix_png))

### 6. XUẤT MODEL (EXPORT) ĐỂ TRIỂN KHAI THỰC TẾ TRÊN CAMERA
Xuất sang định dạng **ONNX** hoặc **TensorRT** để chạy cực nhanh trên PC giám sát tại mục tiêu.

In [ ]:
# Export sang ONNX với tối ưu hóa
best_weight = '/kaggle/working/runs/guard_behavior_model/weights/best.pt'
trained_model = YOLO(best_weight)

onnx_path = trained_model.export(format='onnx', dynamic=True, simplify=True)
print(f"Đã xuất ONNX thành công: {onnx_path}")

### 7. TẢI FILE WEIGHT VỀ MÁY CỤC BỘ
Nén thư mục trọng số để tải về máy và nạp vào hệ thống giám sát camera.

In [ ]:
!zip -r /kaggle/working/guard_behavior_model_weights.zip /kaggle/working/runs/guard_behavior_model/weights/
print("Hoàn tất nén file! Bạn có thể tải file guard_behavior_model_weights.zip ở thanh Output bên phải.")